Case study 2: Applying BPNN to predict credit risk

In [23]:
!pip install imbalanced-learn

In [24]:

import numpy as np
import pandas as pd
import matplotlib as mpl
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [25]:
# Convert discrete data into continuous data
def parse(v,l):
    # v is a string, representing the data to be converted. l is the category information.
    return [1 if i==v else 0 for i in l]

In [26]:
def parseRecord(record):
    result=[ ]
    a1=record['A1']
    for i in parse(a1,('a','b')):
        result.append(i)
    result.append(float(record['A2']))
    result.append(float(record['A3']))
    a4=record['A4']
    for i in parse(a4,('u','y','l','t')):
        result.append(i)
    a5 = record['A5']
    for i in parse(a5,('g','p','gg')):
        result.append(i)
    a6 = record['A6']
    for i in parse(a6, ('c', 'd','cc','i','j','k','m','r','q','w','x','e','aa','ff')):
        result.append(i)
    a7 = record['A7']
    for i in parse(a7, ('v', 'h','bb','j','n','z','dd','ff','o')):
        result.append(i)
    result.append(float(record['A8']))
    a9 = record['A9']
    for i in parse(a9, ('t', 'f')):
        result.append(i)
    a10 = record['A10']
    for i in parse(a10, ('t', 'f')):
        result.append(i)
    result.append(float(record['A11']))
    a12 = record['A12']
    for i in parse(a12, ('t', 'f')):
        result.append(i)
    a13 = record['A13']
    for i in parse(a13, ('g', 'p', 's')):
        result.append(i)
    result.append(float(record['A14']))
    result.append(float(record['A15']))
    return result

In [27]:
def load_dataset_crx(df):
    print(sum(df['A16'] == "+"))
    print(sum(df['A16'] == "-"))
    # Abnormal data filtering
    df = df.replace("?", np.nan).dropna(how='any')
    new_names=['A1_0','A1_1',
               'A2','A3',
               'A4_0','A4_1','A4_2','A4_3',
               'A5_0','A5_1','A5_2',
               'A6_0','A6_1','A6_2','A6_3','A6_4','A6_5','A6_6','A6_7','A6_8','A6_9','A6_10','A6_11','A6_12','A6_13',
               'A7_0','A7_1','A7_2','A7_3','A7_4','A7_5','A7_6','A7_7','A7_8',
               'A8',
               'A9_0','A9_1',
               'A10_0','A10_1',
               'A11',
               'A12_0','A12_1',
               'A13_0','A13_1','A13_2',
               'A14','A15']
    datas=df.apply(lambda x:pd.Series(parseRecord(x),index=new_names),axis=1).values
    N = datas.shape[0]
    # Generate label Y
    data=pd.DataFrame(df,columns=['A16'])

    data.loc[data['A16']=="+"]="positive"
    data.loc[data['A16'] == "-"] = "negative"
    labs=data.values


    unique_labs=np.unique(labs).tolist()

    dic_str2index={ }
    dic_index2str={ }
    for i in range(len(unique_labs)):
        lab_str=unique_labs[i]
        dic_str2index[lab_str]=i
        dic_index2str[i]=lab_str
    
    #labs_onehot is the acutal output
    labs_onehot=np.zeros([N,len(unique_labs)])
    for i in range(N):
        labs_onehot[i,dic_str2index[labs[i][0]]]=1

    data_train,data_test,lab_train_onehot,lab_test_onehot=train_test_split(datas,labs_onehot,test_size=0.1,random_state=0)

    ss = StandardScaler()
    data_train = ss.fit_transform(data_train)
    data_test = ss.transform(data_test)
    return data_train, lab_train_onehot, data_test, lab_test_onehot,dic_index2str

In [28]:
# Define the activation function and loss function, and their derivatives during backpropagation.
def sigmoid(z):
    h=1./(1+np.exp(-z))
    return h
def de_sigmoid(h):
    return h*(1-h)
def loss_L2(o,lab):
    diff=lab-o
    sqrDiff=diff**2
    return 0.5*np.sum(sqrDiff)
def de_loss_L2(o,lab):
    return o-lab

In [29]:
# dim_in: Dimension of input features, list_num_hidden: Number of output nodes per layer, list_act_funs: Activation function per layer, list_de_act_funs: Backpropagation function
def build_net(dim_in,list_num_hidden,list_act_funs,list_de_act_funs):
    layers=[]
    for i in range(len(list_num_hidden)):
        layer={}
        if i ==0:
            layer["w"]=0.2*np.random.randn(dim_in,list_num_hidden[i])-0.1
        else:
            layer["w"]=0.2*np.random.randn(list_num_hidden[i-1],list_num_hidden[i])-0.1
            layer["b"]=0.1*np.ones([1,list_num_hidden[i]])
            layer["act_fun"]=list_act_funs[i]
            layer["de_act_fun"]=list_de_act_funs[i]
            layers.append(layer)
    return layers

In [30]:
# Construct a forward propagation function that returns the input of each layer and the output of the last layer.
def fead_forward(datas,layers):
    input_layers=[]
    for i in range(len(layers)):
        layer=layers[i]
        if i ==0:
            inputs = datas
            z=np.dot(inputs,layer["w"])+layer["b"]
            h=layer['act_fun'](z)
            input_layers.append(inputs)
        else:
            inputs=h
            z=np.dot(inputs,layer["w"])+layer["b"]
            h=layer['act_fun'](z)
            input_layers.append(inputs)
    return input_layers,h

In [31]:
# Backpropagation of error
def updata_wb(datas,labs,layers,loss_fun,de_loss_fun,alpha=0.01):
    inputs,output=fead_forward(datas,layers)
    # Calculate loss
    loss=loss_fun(output,labs)
    # Calculate from back to front
    deltas0=de_loss_fun(output,labs)
    # Calculate the error from back to front.
    deltas=[]
    for i in range(len(layers)):
        index=-i-1
        if i==0:
            delta=deltas0*layers[index]["de_act_fun"](output)
        else:
            h=inputs[index+1]
            delta=np.dot(delta,layers[index+1]["w"].T)*layers[index]["de_act_fun"](h)

        deltas.insert(0,delta)

    # Adjust the weights of each layer using the error.
    for i in range(len(layers)):
        dw=np.dot(inputs[i].T,deltas[i])
        db=np.sum(deltas[i],axis=0,keepdims=True)
        # Gradient descent
        layers[i]["w"]=layers[i]["w"]-alpha*dw
        layers[i]["b"] = layers[i]["b"] - alpha * db

    return layers,loss

In [32]:
# Define a model training performance evaluation function
def test_accuracy(datas,labs_true,layers):
    _,output=fead_forward(datas,layers)
    lab_det=np.argmax(output,axis=1) # Determine the prediction type
    labs_true=np.argmax(labs_true,axis=1) # Target output
    N_error=np.where(np.abs(labs_true-lab_det)>0)[0].shape[0]
     # Calculate the error rate and AUC value
    error_rate=N_error/np.shape(datas)[0]
    roc=roc_auc_score(labs_true,lab_det)
    return error_rate,roc

In [33]:
# Define the main function
if __name__ == "__main__":
    #file_data='crx.data'
    df = pd.read_csv("/Users/tymek/Desktop/crx.csv",header=None,names=['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10','A11','A12','A13','A14','A15','A16'])
    data_train,lab_train_onehot,data_test,lab_test_onehot,dic_index2str=load_dataset_crx(df)
    N,dim_in=np.shape(data_train)
    list_num_hidden=[47,20,2]
    list_act_funs=[sigmoid,sigmoid,sigmoid]
    list_de_act_funs=[de_sigmoid,de_sigmoid,de_sigmoid]

     # Define loss function
    loss_fun = loss_L2
    de_loss_fun=de_loss_L2
    
    layers=build_net(dim_in,list_num_hidden,list_act_funs,list_de_act_funs)

     # Conduct training
    n_epoch=200
    batchsize=4
    N_batch=N  #batchsize
    for i in range(n_epoch):
         # Data shuffling
        rand_index=np.random.permutation(N).tolist()
         # Update the weight for each batch.
        loss_sum=0
        for j in range(N_batch):
            index=rand_index[j*batchsize:(j+1)*batchsize]
            batch_datas=data_train[index]
            batch_labs=lab_train_onehot[index]
            layers,loss=updata_wb(batch_datas,batch_labs,layers,loss_fun,de_loss_fun,alpha=0.01)
            loss_sum=loss_sum+loss

        result=test_accuracy(data_train,lab_train_onehot,layers)
        print("Training output: "+"epoch %d error %.2f%% loss_all %.2f auc %.2f" % (i,result[0]*100,loss_sum,result[1]))

      # Testing
    result=test_accuracy(data_test,lab_test_onehot,layers)
    print("Testing output: " + "epoch %d error %.2f%% loss_all %.2f auc %.2f" % (i, result[0] * 100, loss_sum, result[1]))

307
383
Training output: epoch 0 error 44.46% loss_all 156.26 auc 0.52
Training output: epoch 1 error 28.62% loss_all 131.80 auc 0.69
Training output: epoch 2 error 16.18% loss_all 117.43 auc 0.83
Training output: epoch 3 error 15.67% loss_all 104.15 auc 0.84
Training output: epoch 4 error 14.14% loss_all 92.61 auc 0.86
Training output: epoch 5 error 13.46% loss_all 83.46 auc 0.86
Training output: epoch 6 error 12.78% loss_all 76.48 auc 0.87
Training output: epoch 7 error 13.12% loss_all 71.50 auc 0.87
Training output: epoch 8 error 12.44% loss_all 67.64 auc 0.88
Training output: epoch 9 error 11.93% loss_all 64.77 auc 0.88
Training output: epoch 10 error 11.93% loss_all 62.55 auc 0.88
Training output: epoch 11 error 11.58% loss_all 60.81 auc 0.89
Training output: epoch 12 error 11.75% loss_all 59.44 auc 0.88
Training output: epoch 13 error 11.75% loss_all 58.36 auc 0.88
Training output: epoch 14 error 11.41% loss_all 57.41 auc 0.89
Training output: epoch 15 error 11.24% loss_all 56.64